# Membangun Sistem Tanya-Jawab Sederhana dengan TF-IDF + Sastrawi

Notebook ini adalah proyek praktik: mesin pencari FAQ untuk toko furnitur fiktif "Emerald Mabel", menggunakan TF-IDF + stemming Bahasa Indonesia (Sastrawi) untuk mencocokkan pertanyaan user dengan database FAQ.

Termasuk audit perbandingan **Euclidean distance vs Cosine similarity** untuk menentukan metrik pencarian yang lebih andal.

> Setup library (scikit-learn, Sastrawi, joblib) — lihat `requirements.txt`.


In [1]:
import json
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import joblib

np.set_printoptions(precision=4, suppress=True)
print("Semua library berhasil diimpor.")

Semua library berhasil diimpor.


## 4. Praktik: Sistem Tanya-Jawab (FAQ Search Engine)

Proyek ini membangun mesin pencari FAQ sederhana untuk toko furnitur fiktif "Emerald Mabel", menggunakan TF-IDF + stemming Bahasa Indonesia (Sastrawi) untuk mencocokkan pertanyaan user dengan pertanyaan di database FAQ.

> Dataset FAQ ini sepenuhnya **fiktif/contoh** (bukan data toko sungguhan), dipakai murni untuk demonstrasi teknik.

### 4.1 Dataset FAQ

In [2]:
qa_corpus = [
    {"question": "Apakah Emerald Mabel menyediakan pengiriman untuk semua produk?", "answer": "Ya, kami menyediakan layanan pengiriman untuk semua produk kami. Biaya pengiriman dapat bervariasi tergantung lokasi."},
    {"question": "Berapa lama waktu pengiriman biasanya?", "answer": "Waktu pengiriman tergantung pada lokasi pengiriman. Pengiriman lokal biasanya memakan waktu 3-5 hari kerja, sedangkan pengiriman luar kota bisa mencapai 7-10 hari kerja."},
    {"question": "Apakah saya bisa mengembalikan produk jika tidak puas?", "answer": "Tentu, Anda bisa mengembalikan produk dalam waktu 14 hari setelah menerima produk, asalkan produk dalam kondisi yang sama saat diterima."},
    {"question": "Bagaimana cara mengajukan pengembalian barang?", "answer": "Silakan hubungi customer service kami melalui telepon atau email dengan menyertakan bukti pembelian dan alasan pengembalian barang."},
    {"question": "Apakah ada garansi untuk produk dari Emerald Mabel?", "answer": "Ya, kami memberikan garansi selama 1 tahun untuk setiap produk dengan cacat pabrik. Garansi ini tidak termasuk kerusakan karena penggunaan."},
    {"question": "Bagaimana cara melacak pesanan saya?", "answer": "Anda akan menerima email berisi nomor pelacakan setelah pesanan dikirim. Gunakan nomor tersebut untuk melacak status pengiriman di situs kami."},
    {"question": "Apakah saya bisa memesan produk custom di Emerald Mabel?", "answer": "Ya, kami menerima pesanan custom untuk furnitur. Silakan hubungi tim kami untuk diskusi lebih lanjut mengenai spesifikasi yang Anda inginkan."},
    {"question": "Metode pembayaran apa saja yang diterima?", "answer": "Kami menerima pembayaran melalui transfer bank, kartu kredit, dan pembayaran digital seperti e-wallet."},
    {"question": "Apakah ada diskon untuk pembelian dalam jumlah besar?", "answer": "Ya, kami menyediakan diskon untuk pembelian dalam jumlah besar. Silakan hubungi customer service kami untuk informasi lebih lanjut."},
    {"question": "Apakah Emerald Mabel memiliki showroom yang bisa dikunjungi?", "answer": "Ya, kami memiliki showroom yang dapat Anda kunjungi. Silakan cek alamat dan jam operasional kami di situs web."},
    {"question": "Bagaimana cara merawat furnitur kayu agar tahan lama?", "answer": "Untuk menjaga ketahanan furnitur kayu, hindari paparan sinar matahari langsung dan bersihkan secara berkala dengan kain lembut."},
    {"question": "Apakah tersedia layanan perakitan untuk produk yang saya beli?", "answer": "Ya, kami menyediakan layanan perakitan untuk beberapa produk. Mohon tanyakan kepada customer service untuk produk yang memenuhi syarat."},
    {"question": "Bisakah saya membatalkan pesanan yang sudah dibuat?", "answer": "Anda bisa membatalkan pesanan sebelum pesanan dikirim. Jika pesanan sudah dikirim, Anda harus menunggu barang sampai dan mengajukan pengembalian."},
    {"question": "Bagaimana jika produk yang diterima dalam kondisi rusak?", "answer": "Jika produk yang Anda terima dalam kondisi rusak, segera hubungi customer service kami untuk klaim penggantian atau pengembalian."},
    {"question": "Apakah Emerald Mabel menyediakan layanan pengiriman ke luar negeri?", "answer": "Saat ini kami hanya melayani pengiriman dalam negeri. Namun, Anda bisa menghubungi customer service untuk kemungkinan lainnya."},
    {"question": "Apakah saya bisa mengatur jadwal pengiriman?", "answer": "Ya, Anda bisa mengatur jadwal pengiriman yang sesuai. Silakan berikan preferensi jadwal kepada customer service kami saat melakukan pemesanan."},
    {"question": "Apakah ada biaya tambahan untuk pengiriman ke daerah terpencil?", "answer": "Ya, pengiriman ke daerah terpencil mungkin dikenakan biaya tambahan. Customer service kami akan memberi tahu biaya tambahan jika ada."},
    {"question": "Bagaimana cara mendapatkan katalog produk Emerald Mabel?", "answer": "Anda bisa mendapatkan katalog produk dengan mengunjungi situs web kami atau menghubungi customer service untuk mengirimkan katalog ke email Anda."},
    {"question": "Apa yang harus saya lakukan jika pesanan saya belum sampai?", "answer": "Jika pesanan Anda belum sampai dalam waktu yang dijanjikan, silakan hubungi customer service kami untuk bantuan lebih lanjut."},
    {"question": "Apakah ada promo khusus untuk pelanggan baru?", "answer": "Ya, kami sering mengadakan promo untuk pelanggan baru. Kunjungi situs web kami atau hubungi customer service untuk informasi promo terbaru."},
]

print(f"Total pasangan tanya-jawab: {len(qa_corpus)}")

Total pasangan tanya-jawab: 20


### 4.2 Preprocessing: Lowercase + Hapus Tanda Baca + Stemming (Sastrawi)

> **Catatan koreksi:** di kode aslinya, fungsi ini diberi nama komentar "Lemmatize", padahal yang sebenarnya dilakukan `stemmer.stem()` dari Sastrawi adalah **stemming**, bukan lemmatization. Di notebook ini penamaannya diperbaiki menjadi `preprocess_text` yang lebih akurat.

In [3]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    words = text.split()
    stemmed_words = [stemmer.stem(word) for word in words]
    return ' '.join(stemmed_words)

contoh = qa_corpus[0]["question"]
print("Sebelum:", contoh)
print("Sesudah:", preprocess_text(contoh))

Sebelum: Apakah Emerald Mabel menyediakan pengiriman untuk semua produk?


Sesudah: apakah emerald mabel sedia kirim untuk semua produk


### 4.3 Melatih TF-IDF Vectorizer

In [4]:
questions = [item["question"] for item in qa_corpus]
answers = [item["answer"] for item in qa_corpus]

preprocessed_questions = [preprocess_text(q) for q in questions]
preprocessed_answers = [preprocess_text(a) for a in answers]
combined_corpus = preprocessed_questions + preprocessed_answers

vectorizer = TfidfVectorizer()
vectorizer.fit(combined_corpus)

question_vectors = vectorizer.transform(preprocessed_questions)

print(f"Ukuran vocabulary: {len(vectorizer.get_feature_names_out())} kata")
print(f"Jumlah vektor pertanyaan: {question_vectors.shape[0]}")

joblib.dump(vectorizer, "vectorizer.joblib")
print("Vectorizer disimpan ke vectorizer.joblib")

Ukuran vocabulary: 179 kata
Jumlah vektor pertanyaan: 20
Vectorizer disimpan ke vectorizer.joblib


### 4.4 Fungsi Pencarian: Audit Euclidean Distance vs Cosine Similarity

Kode aslinya memakai **Euclidean distance** dengan threshold tetap (`THR = 1`) untuk menentukan kecocokan. Di sini kita bandingkan langsung dengan **Cosine similarity**, yang lebih umum dipakai untuk pencarian teks berbasis TF-IDF.

In [5]:
THR = 1.0

def find_best_match(user_question, metric="euclidean"):
    processed = preprocess_text(user_question)
    user_vector = vectorizer.transform([processed]).toarray()

    best_score = float("inf") if metric == "euclidean" else -1
    best_item = None

    for i, item in enumerate(qa_corpus):
        qv = question_vectors[i].toarray()
        if metric == "euclidean":
            score = euclidean_distances(user_vector, qv)[0][0]
            is_better = score < best_score and score <= THR
        else:
            score = cosine_similarity(user_vector, qv)[0][0]
            is_better = score > best_score
        if is_better:
            best_score = score
            best_item = item

    return best_item, best_score

In [6]:
test_queries = [
    "berapa lama pengiriman barang",
    "cara mengembalikan barang yang rusak",
    "apakah ada diskon",
    "resep nasi goreng enak",  # pertanyaan di luar topik FAQ, sengaja untuk uji ketahanan
]

for q in test_queries:
    print("=" * 70)
    print("Pertanyaan user:", q)

    item_euc, score_euc = find_best_match(q, metric="euclidean")
    if item_euc:
        print(f"[Euclidean] Cocok: \"{item_euc['question']}\"  (distance={score_euc:.4f})")
    else:
        print(f"[Euclidean] Tidak ada yang cocok di bawah threshold {THR}")

    item_cos, score_cos = find_best_match(q, metric="cosine")
    if item_cos:
        print(f"[Cosine]    Cocok: \"{item_cos['question']}\"  (similarity={score_cos:.4f})")

Pertanyaan user: berapa lama pengiriman barang
[Euclidean] Cocok: "Berapa lama waktu pengiriman biasanya?"  (distance=0.8294)
[Cosine]    Cocok: "Berapa lama waktu pengiriman biasanya?"  (similarity=0.6560)
Pertanyaan user: cara mengembalikan barang yang rusak
[Euclidean] Cocok: "Bagaimana cara mengajukan pengembalian barang?"  (distance=0.9124)
[Cosine]    Cocok: "Bagaimana cara mengajukan pengembalian barang?"  (similarity=0.5838)
Pertanyaan user: apakah ada diskon
[Euclidean] Cocok: "Apakah ada diskon untuk pembelian dalam jumlah besar?"  (distance=0.8952)
[Cosine]    Cocok: "Apakah ada diskon untuk pembelian dalam jumlah besar?"  (similarity=0.5993)
Pertanyaan user: resep nasi goreng enak
[Euclidean] Cocok: "Bagaimana cara merawat furnitur kayu agar tahan lama?"  (distance=1.0000)
[Cosine]    Cocok: "Apakah Emerald Mabel menyediakan pengiriman untuk semua produk?"  (similarity=0.0000)


### 4.5 Temuan Audit

Dari hasil run di atas, ada satu temuan konkret yang layak dicatat:

**Query di luar topik (`"resep nasi goreng enak"`) tetap "cocok" secara Euclidean** — distance-nya persis jatuh di batas threshold (`1.0000`), sehingga sistem tetap mengembalikan jawaban meskipun pertanyaannya sama sekali tidak berhubungan dengan FAQ furnitur (ini **false positive**). Sementara itu, cosine similarity untuk query yang sama menghasilkan `0.0000` — sinyal yang jauh lebih jelas dan tegas bahwa tidak ada kecocokan sama sekali.

Untuk ketiga query yang relevan (soal pengiriman, pengembalian, dan diskon), kedua metrik sama-sama berhasil menemukan pertanyaan yang tepat — tapi jarak Euclidean-nya (0.83–0.91) juga berada cukup dekat dengan angka threshold 1.0. Ini menunjukkan ambang batas `THR = 1` pada kode asli cukup rapuh: query yang benar-benar relevan bisa saja gagal lolos threshold hanya karena sedikit perbedaan panjang atau pilihan kata, sementara query yang sama sekali tidak relevan justru bisa lolos karena kebetulan jaraknya pas di batas.

**Kesimpulan:** untuk pencarian TF-IDF seperti ini, cosine similarity memberi sinyal relevansi yang lebih tegas dan tidak bergantung pada satu angka ambang batas yang rawan meleset, karena ia mengukur kemiripan *arah* vektor (relevansi konten) dan tidak terlalu terpengaruh oleh panjang teks — sementara Euclidean distance sensitif terhadap magnitudo vektor, sehingga rentan salah pada teks dengan panjang atau kepadatan kata yang berbeda-beda.

## 5. Ringkasan

| Teknik | Kelebihan | Kekurangan | Cocok untuk |
|---|---|---|---|
| Bag of Words | Sederhana, cepat | Abaikan urutan & makna kata, dimensi besar | Baseline klasifikasi teks sederhana |
| TF-IDF | Memberi bobot relevansi kata | Masih abaikan urutan & konteks semantik | Pencarian dokumen, klasifikasi teks |
| Word2Vec | Menangkap kemiripan makna antar kata | Butuh korpus besar untuk hasil bermakna | Tugas NLP yang butuh pemahaman semantik |

Untuk sistem pencarian FAQ seperti proyek di atas, kombinasi **TF-IDF + Sastrawi (stemming) + cosine similarity** terbukti lebih tahan terhadap pertanyaan di luar topik dibanding pendekatan Euclidean distance dengan threshold tetap.


---

**Sumber materi:** rubythalib.ai AI Engineer Bootcamp — modul NLP (mentor: Muhammad Ikhwan Fathulloh).
**Catatan:** contoh kode telah dijalankan ulang dan diverifikasi; sebagian penamaan variabel dan pendekatan (lihat bagian 4.2 dan 4.5) dikoreksi/diaudit dari versi aslinya.


---

**Sumber materi:** rubythalib.ai AI Engineer Bootcamp — modul NLP (mentor: Muhammad Ikhwan Fathulloh).
**Catatan:** dataset FAQ di notebook ini sepenuhnya fiktif (contoh), dipakai murni untuk demonstrasi teknik. Kode `train.py` dan `app.py` versi production-style (di luar notebook) tersedia di root repo ini.
